# Orbit Wars: PPO Continuous Training Pipeline

This notebook is designed to train the OrbitsNet actor-critic model using Kaggle's compute resources. 
It copies your codebase into the writable `/kaggle/working` directory, installs dependencies, pretrains the value head on replays, and then launches into an infinite self-play loop.

The loop will continuously save `last.pt` and `best.pt` directly to `/kaggle/working/OrbitsWar/rl/`.


In [ ]:
import os, shutil, sys

WORK_DIR = "/kaggle/working/OrbitsWar"

# ── Auto-detect dataset mount path (handles nested /datasets/{user}/{slug}/ paths) ──
def find_dataset_path():
    base = "/kaggle/input"
    if not os.path.exists(base):
        return None
    for root, dirs, files in os.walk(base):
        dirs.sort()
        if "main.py" in files and "records" in dirs:
            print(f"Found OrbitsWar dataset at: {root}")
            return root
    print("Did not find main.py + records/ anywhere under /kaggle/input/")
    print("Full tree:")
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > 4:
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root)}/  files={files[:5]}")
    return None

INPUT_DIR = find_dataset_path()
if INPUT_DIR is None:
    raise RuntimeError(
        "Could not find OrbitsWar dataset. See tree printout above.\n"
        "Make sure you attached the dataset to this notebook (Add data -> Your datasets)."
    )

!pip install -q kaggle-environments

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(INPUT_DIR, WORK_DIR)
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

print(f"Input:   {INPUT_DIR}")
print(f"WorkDir: {WORK_DIR}")
print("Contents:", sorted(os.listdir(WORK_DIR)))

## Phase A: Pretrain on Replays (Value Head Calibration)
This runs BCE and offline TD on your parsed Kaggle replays to give the value function a head start before self-play.

In [ ]:
import os, subprocess
if os.path.exists("rl/last.pt") or os.path.exists("rl/best.pt"):
    print("Checkpoint found — skipping Phase A (resuming from existing weights)")
else:
    print("No checkpoint — running Phase A (replay pretraining)...")
    subprocess.run(["python", "rl/replay_trainer.py"], check=True)

## Phase B: Continuous Self-Play
This cell will run indefinitely (100,000 rounds). It will actively overwrite `last.pt` each round and update `best.pt` when a new high score is achieved. You can stop this cell manually at any time without losing your progress.

In [ ]:
!python rl/self_play_trainer.py

---
## Phase C: Export Weights
Converts `best.pt` → `rl/rl_value_fn.py` (pure-Python numpy arrays, no PyTorch at submission time).
Run this after stopping self-play.

In [ ]:
!python rl/export_weights.py